# Election Bloc Change Prediction Project
## Notebook 03 — Historical EDA and rolling baselines

Evaluates persistence and simple historical-swing baselines over multiple expanding-window temporal backtests. K24→K25 remains untouched.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
PANEL=INTERIM_DIR/"election_transition_panel.csv"; TYPES=RAW_DIR/"Locality_Types.xlsx"
if not PANEL.exists() and (REPO_ROOT/".git").exists(): subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
panel=pd.read_csv(PANEL,dtype={"locality_symbol":"string"},low_memory=False)
MODELED_BLOCS=["Right","Center_Left","Haredi","Arab"]
# Locality type is optional but strongly recommended.
if TYPES.exists():
    raw_type=pd.read_excel(TYPES)
    raw_type.columns=[re.sub(r"\s+"," ",str(c).replace("\ufeff","").strip()) for c in raw_type.columns]
    sc=next(c for c in raw_type.columns if "סמל" in c); tc=next(c for c in raw_type.columns if c!=sc and ("type" in c.lower() or "סוג" in c))
    lt=pd.DataFrame({"locality_symbol":pd.to_numeric(raw_type[sc],errors="coerce").astype("Int64").astype("string"),"locality_type":raw_type[tc].astype("string")}).dropna().drop_duplicates("locality_symbol")
    panel=panel.merge(lt,on="locality_symbol",how="left")
else: panel["locality_type"]="Unknown"
panel["locality_type"]=panel.locality_type.fillna("Unknown")
# Keep the existing broad grouping convention.
panel["analysis_group"]=np.where(panel.locality_type.astype(str).str.contains("Arab|Bedouin|Druze|Non-Jewish|ערב",case=False,regex=True),"Arab/Non-Jewish","Non-Arab")
print(panel.groupby(["transition_id","analysis_group"]).size().unstack(fill_value=0))

## Rolling temporal backtests

In [ ]:
CURRENT=[f"current_{b}_pct" for b in MODELED_BLOCS]; PREVIOUS=[f"previous_{b}_pct" for b in MODELED_BLOCS]
ordered=[f"K{n}_to_K{n+1}" for n in range(16,25)]
DEV=[t for t in ordered if t!="K24_to_K25"]
TEST_FOLDS=DEV[3:]  # at least three earlier transitions before every test fold

def metrics(data,pred,name,fold):
    y=data[CURRENT].to_numpy(float); p=np.asarray(pred,float); ae=np.abs(y-p); row=ae.mean(1);w=pd.to_numeric(data.current_valid_votes,errors="coerce").fillna(0).to_numpy()
    return {"transition_id":fold,"model":name,"rows":len(data),"mae":float(ae.mean()),"median_row_mae":float(np.median(row)),"weighted_mae":float(np.average(row,weights=w)) if w.sum()>0 else np.nan}
rows=[]; by_group=[]; by_bloc=[]
for fold in TEST_FOLDS:
    idx=ordered.index(fold); train_ids=ordered[:idx]; tr=panel[panel.transition_id.isin(train_ids)]; te=panel[panel.transition_id.eq(fold)]
    pred_p=te[PREVIOUS].to_numpy(float)
    global_delta=tr[[f"delta_{b}_pct" for b in MODELED_BLOCS]].mean().to_numpy(float); pred_g=np.clip(pred_p+global_delta,0,None);pred_g=pred_g/pred_g.sum(1,keepdims=True)*100
    type_delta=tr.groupby("locality_type")[[f"delta_{b}_pct" for b in MODELED_BLOCS]].mean(); pred_t=[]
    for _,r in te.iterrows():
        d=type_delta.loc[r.locality_type].to_numpy(float) if r.locality_type in type_delta.index else global_delta
        z=np.clip(r[PREVIOUS].to_numpy(float)+d,0,None);pred_t.append(z/z.sum()*100)
    preds={"Previous-election persistence":pred_p,"Global historical swing":pred_g,"Locality-type historical swing":np.vstack(pred_t)}
    for name,pred in preds.items():
        rows.append(metrics(te,pred,name,fold))
        for g,sub in te.groupby("analysis_group"):
            loc=te.index.get_indexer(sub.index); by_group.append({**metrics(sub,pred[loc],name,fold),"analysis_group":g})
        y=te[CURRENT].to_numpy(float)
        for j,b in enumerate(MODELED_BLOCS): by_bloc.append({"transition_id":fold,"model":name,"bloc":b,"rows":len(te),"mae":float(np.abs(y[:,j]-pred[:,j]).mean())})
rolling=pd.DataFrame(rows); rolling.groupby("model")[["mae","weighted_mae"]].agg(["mean","std","count"]).sort_values(("mae","mean"))

## Save

In [ ]:
paths={"panel":INTERIM_DIR/"election_transition_panel_with_type.csv","rolling":TABLES_DIR/"rolling_baseline_comparison.csv","group":TABLES_DIR/"rolling_baseline_by_group.csv","bloc":TABLES_DIR/"rolling_baseline_by_bloc.csv","summary":SUMMARIES_DIR/"notebook_03_summary.json"}
panel.to_csv(paths["panel"],index=False,encoding="utf-8-sig");rolling.to_csv(paths["rolling"],index=False,encoding="utf-8-sig");pd.DataFrame(by_group).to_csv(paths["group"],index=False,encoding="utf-8-sig");pd.DataFrame(by_bloc).to_csv(paths["bloc"],index=False,encoding="utf-8-sig")
agg=rolling.groupby("model").mae.agg(["mean","std","count"]).sort_values("mean")
summary={"notebook":"03_eda_and_baselines","development_transitions":DEV,"temporal_backtest_folds":TEST_FOLDS,"best_baseline":agg.index[0],"baseline_summary":agg.reset_index().to_dict("records"),"final_test_status":"K24_to_K25 locked"}
paths["summary"].write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))